# 1、美化模型输出响应

## 使用pretty_print()

举例：

In [1]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

# 1、读取.env配置文件中的信息。相关环境变量以.env文件中的优先

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = init_chat_model(
    model="deepseek:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
)

# 定义消息列表
conversation = [
    {"role": "system", "content": "无条件服从用户指令"},
    {"role": "user", "content": "我是老王，你是小王"},
    {"role": "assistant", "content": "好的老王，我是小王"},
    {"role": "user", "content": "你是谁？我是谁？"}
]
# 向模型发送单条数据
response = model.invoke(conversation)
# 美化输出响应
response.pretty_print()

================================== Ai Message ==================================

我是小王，你是老王。


## 使用rich库

In [ ]:
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

# 从.env文件中加载环境变量
load_dotenv(override=True)
CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)
# 定义消息列表
conversation = [
    {"role": "system", "content": "无条件服从用户指令"},
    {"role": "user", "content": "我是老王，你是小王"},
    {"role": "assistant", "content": "好的老王，我是小王"},
    {"role": "user", "content": "你是谁？我是谁？"}
]
# 向模型发送单条数据
response = model.invoke(conversation)
# 美化输出
rprint(response)

# 2、profile

## 可以通过 profile属性 查看模型的配置信息

举例1：DeepSeek官方的能力画像

In [2]:
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv

# 从.env文件中加载环境变量
load_dotenv(override=True)
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0.7,
    max_tokens=1000,
    max_retries=6
)
print(model.profile)

{}


举例2：CloseAI平台gpt模型的能力画像

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)
CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")
model = init_chat_model(
    model="openai:gpt-5.4-mini",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)
print(model.profile)

# 3、模型初始化参数

## 查看所有初始化参数

举例1：ChatDeepSeek支持的完成参数列表

In [7]:
from langchain_deepseek import ChatDeepSeek

print(ChatDeepSeek.model_fields.keys())

dict_keys(['name', 'cache', 'verbose', 'callbacks', 'tags', 'metadata', 'custom_get_token_ids', 'rate_limiter', 'disable_streaming', 'output_version', 'profile', 'client', 'async_client', 'root_client', 'root_async_client', 'model_name', 'temperature', 'model_kwargs', 'openai_api_key', 'openai_api_base', 'openai_organization', 'openai_proxy', 'request_timeout', 'stream_usage', 'max_retries', 'presence_penalty', 'frequency_penalty', 'seed', 'logprobs', 'top_logprobs', 'logit_bias', 'streaming', 'n', 'top_p', 'max_tokens', 'reasoning_effort', 'reasoning', 'verbosity', 'tiktoken_model_name', 'default_headers', 'default_query', 'http_client', 'http_async_client', 'stop', 'extra_body', 'include_response_headers', 'disabled_params', 'context_management', 'include', 'service_tier', 'store', 'truncation', 'use_previous_response_id', 'use_responses_api', 'api_key', 'api_base'])


举例2：查看ChatOpenAI支持的完整参数列表

In [ ]:
from langchain_openai import ChatOpenAI

print(ChatOpenAI.model_fields.keys())


举例3：查看init_chat_model的某model_provider支持的完整参数列表

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)
# 1. 实例化一个你感兴趣的模型对象
# 即使不传入具体 key，通常也能初始化成功
temp_model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
)
# 2. 现在它已经是一个具体的 ChatDeepSeek 对象了
# 你可以使用你熟悉的 .model_fields.keys()
print(temp_model.model_fields.keys())

举例4：可以查看每个字段的属性

In [ ]:
from langchain_deepseek import ChatDeepSeek

for name, field in ChatDeepSeek.model_fields.items():
    print(name)
    print(" annotation:", field.annotation)
    print(" default:", field.default)
    print(" description:", getattr(field, "description", None))
    print(" alias:", field.alias)
    print()

# 4、高级与特定扩展参数

这类参数通常用于特定场景，或为了保持与 OpenAI 协议的兼容性而存在。

## 参数：model_kwargs

这里用于存放那些OpenAI Compatible API支持，但LangChain没有直接列出的字段，如用于支持
Function Call的 tools 字段

In [8]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)
model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    model_kwargs={"tools": [
        {
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Get weather of a location, the user should supply a location first.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA",
                        }
                    },
                    "required": ["location"]
                },
            }
        },
    ]}
)
# 向模型发送单条数据
response = model.invoke("你好，今天北京的天气如何")
# 打印响应
rprint(response)


AIMessage(
    content='你好！我来帮你查一下今天北京的天气。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'The user is asking about the weather in Beijing today. I should use the get_weather 
tool to check.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 70,
            'prompt_tokens': 319,
            'total_tokens': 389,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 22,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 319
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '17683d93-8995-4c68-9d20-6089448948b5',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0f587-86de-7e80-98ba-6a7a9b879c90-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'location': '北京'},
            'id': 'call_00_HtyB6G5xEr4BwLNPhUza9862',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 319,
        'output_tokens': 70,
        'total_tokens': 389,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 22}
    }
)

## 参数：extra_body

这里用于存放模型厂商基于OpenAI API协议扩展的字段


In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)
model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body={"thinking": {"type": "enabled"}},
)
# 向模型发送单条数据
response = model.invoke("你好，一句话回答")
# 打印响应
rprint(response)


# 4、模型调用（invoke、stream、batch等）中config参数

## config参数：允许在调用模型时，动态地配置和控制模型的行为，而无需在初始化时就固定所有参数，这为应用带来了极大的灵活性和可维护性。

说明如下：
+ config中参数 run_name 、 tags 、 callbacks 主要用在LangSmith中，用于追踪、筛选和调试。
+ metadata 可以配置用户指定的一些信息，在工作流开发中，当整个流程被包装为Runnable链
时，可以将这些参数传递给后续的链节点使用。
+ configurable 中可配置的参数与 init_chat_model 初始化模型参数一样，与在初始化模型时设置
的参数（如 temperature=0.7）的关键区别在于：
  - init_chat_model初始化参数：模型的 默认设置 ，适用于该模型实例的大部分场景。
  - 运行时 config： 单次调用的特定设置 ，优先级更高，针对本次调用进行的特殊调整。

举例

In [ ]:
model.invoke(
    "你好",
    config={
        "run_name": "...",  # 在LangSmith中这次运行会显示为指定名称
        "tags": ["test", "development"],  # 打上标签便于分类查找
        "metadata": {"user_id": "123"},  # 记录用户ID
        "callbacks": [custom_handler],  # 启用自定义回调函数
        "configurable": {
            "model": "deepseek-reasoner",  # 配置模型参数
            "temperature": 0.7,  # 配置温度参数
            "max_tokens": 100  # 配置最大令牌数
        }
    }
)


#### 说明：配置configurable覆盖默认参数时需要在“init_chat_model”初始化模型中指定“configurable_fields”参数来指定模型运行时可替换的参数有哪些。
举例2

In [9]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
# 1. 初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    temperature=0.2,
    max_tokens=500,
    # 指定可调整参数
    configurable_fields=("model", "model_provider", "temperature",
                         "max_tokens"),
)

# 2. 准备 config 字典
config = {
    "run_name": "joke_generation",  # 在LangSmith中这次运行会显示为
    "joke_generation"
    "tags": ["tag1", "tag2"],  # 打上标签便于分类查找
    "metadata": {"user_id": "123"},  # 记录用户ID
    "configurable": {
        "model": "deepseek-v4-pro",  # 配置模型参数
        "model_provider": "openai",  # 配置模型提供商参数
        "temperature": 0.7,  # 配置温度参数
        "max_tokens": 1000  # 配置最大令牌数
    }
}

# 3. 调用模型并传入config
response = model.invoke(
    "1 + 2 = ？",
    config=config
)
rprint(response)


AIMessage(
    content='3',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 49,
            'prompt_tokens': 90,
            'total_tokens': 139,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 47,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 90
        },
        'model_provider': 'openai',
        'model_name': 'deepseek-v4-pro',
        'system_fingerprint': 'a307abda487cd1b463329ccb945ce396',
        'id': '8030f2a4-59b9-4eab-8618-433782036ba6',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0f58f-3276-7202-ac28-af098536955d-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 90,
        'output_tokens': 49,
        'total_tokens': 139,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 47}
    }
)